# Batch Token-Budget Distribution Walkthrough

This executable example follows one small batch from raw completed sessions through eligibility, allocation, deterministic token-constrained selection, paced scheduling, telemetry, checkpointing, and replay. Planning is label-blind: no outcome or quality signal participates in membership.

## 1. Import the planner

Run the notebook from the repository root. The setup adds the repository root to the Python import path when necessary.

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'random_sampling').is_dir():
    repo_root = next(parent for parent in repo_root.parents if (parent / 'random_sampling').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from random_sampling.budget_distribution import (
    AllocationConfig, BatchCheckpoint, BudgetDeductions, FairnessState, JsonCheckpointStore,
    SessionDemand, build_batch_plan, build_batch_telemetry, build_eligible_frame,
    calculate_batch_budget, resolve_batch_window,
)

print(f'Repository root: {repo_root}')

Repository root: C:\Users\stangoodwin\singlenotebooks-random-budget-distribution


## 2. Freeze a window and calculate its effective budget

A successful watermark and fixed cutoff make the canonical half-open window. The actual elapsed interval is three minutes, but the catch-up cap is 0.05 minutes, yielding 1,000 nominal tokens at 20,000 TPM. Explicit safety, retry, and output reserves leave 900 tokens for selection.

In [2]:
previous_watermark = datetime(2026, 8, 6, 12, 0, tzinfo=timezone.utc)
cutoff = previous_watermark + timedelta(minutes=3)
window = resolve_batch_window(
    previous_successful_watermark=previous_watermark, cutoff=cutoff,
    lookback=timedelta(minutes=2), max_catchup_minutes=0.05,
)
budget = calculate_batch_budget(
    window=window,
    deductions=BudgetDeductions(safety_tokens=50, retry_tokens=25, output_tokens=25),
)
print({'elapsed_minutes': window.elapsed_minutes, 'clamped_minutes': window.clamped_minutes,
       'nominal_tokens': budget.nominal_tokens, 'effective_tokens': budget.effective_tokens})

{'elapsed_minutes': 3.0, 'clamped_minutes': 0.05, 'nominal_tokens': 1000, 'effective_tokens': 900}


## 3. Create raw completed-session demand

A demand record contains identity, timestamps, version, and estimated input/output cost only. The source list intentionally has one duplicate, an unseen late arrival, an already processed late arrival, and a future record so frame filtering is visible.

In [3]:
def make_session(session_id, tenant, agent, completed_offset, ingested_offset, cost):
    return SessionDemand(
        tenant_id=tenant, agent_id=agent, session_id=session_id, session_version='v1',
        completed_at=previous_watermark + timedelta(minutes=completed_offset),
        ingested_at=previous_watermark + timedelta(minutes=ingested_offset),
        estimated_input_tokens=cost - 20, expected_output_tokens=20,
    )

source_sessions = [
    make_session('alpha-1', 'contoso', 'support', 1, 1, 300),
    make_session('alpha-1', 'contoso', 'support', 1, 2, 300),  # duplicate
    make_session('alpha-2', 'contoso', 'sales', 2, 2, 250),
    make_session('beta-1', 'fabrikam', 'support', 2, 2, 350),
    make_session('late-1', 'fabrikam', 'support', -1, 1, 150),
    make_session('processed-1', 'contoso', 'support', -1, 1, 100),
    make_session('future-1', 'contoso', 'sales', 5, 5, 100),
]
processed_keys = {source_sessions[5].dedup_key}
[(item.dedup_key, item.total_cost_tokens) for item in source_sessions]

[('contoso/support/alpha-1/v1', 300),
 ('contoso/support/alpha-1/v1', 300),
 ('contoso/sales/alpha-2/v1', 250),
 ('fabrikam/support/beta-1/v1', 350),
 ('fabrikam/support/late-1/v1', 150),
 ('contoso/support/processed-1/v1', 100),
 ('contoso/sales/future-1/v1', 100)]

## 4. Freeze an eligible frame

The frame deduplicates exact session versions, excludes already processed records, admits unseen late records through the source lookback, and creates canonical membership and frame hashes for audit and retry.

In [4]:
frame = build_eligible_frame(
    source_sessions=source_sessions, window=window, processed_session_keys=processed_keys
)
print({
    'canonical_count': frame.canonical_count,
    'lookback_admitted_count': frame.lookback_admitted_count,
    'duplicate_count': frame.duplicate_count,
    'processed_count': frame.processed_count,
    'frame_hash': frame.frame_hash[:12],
    'membership_hash': frame.membership_hash[:12],
})
[(item.dedup_key, item.total_cost_tokens) for item in frame.sessions]

{'canonical_count': 4, 'lookback_admitted_count': 1, 'duplicate_count': 1, 'processed_count': 1, 'frame_hash': '52c1c8cb3e6e', 'membership_hash': '296a170f11b6'}


[('fabrikam/support/late-1/v1', 150),
 ('contoso/support/alpha-1/v1', 300),
 ('contoso/sales/alpha-2/v1', 250),
 ('fabrikam/support/beta-1/v1', 350)]

## 5. Allocate grants, choose whole sessions, and produce a schedule

The planner protects feasible tenant and agent floors, uses prior fairness deficits as priority, ranks sessions stably from the seed and frame hash, packs whole sessions into grants, then constructs a rolling-TPM dispatch schedule. Indivisible slack remains visible instead of selecting part of a session.

In [5]:
fairness_in = FairnessState(
    tenant_deficit_tokens={'fabrikam': 100},
    agent_deficit_tokens={'fabrikam/support': 100},
)
plan = build_batch_plan(
    pipeline_id='walkthrough', batch_id='example-batch-001', seed='walkthrough-seed-2026-08-06',
    window=window, budget=budget, frame=frame, fairness_state=fairness_in,
    allocation_config=AllocationConfig(tenant_floor_tokens=100, agent_floor_tokens=50),
)
print({
    'effective_budget': plan.budget.effective_tokens,
    'planned_usage': plan.planned_usage_tokens,
    'slack': plan.slack_tokens,
    'selected_sessions': len(plan.selection.selected),
    'redistribution_rounds': plan.reallocation_rounds,
})
assert plan.planned_usage_tokens <= plan.budget.effective_tokens

{'effective_budget': 900, 'planned_usage': 750, 'slack': 150, 'selected_sessions': 3, 'redistribution_rounds': 2}


In [6]:
import pandas as pd

grants = pd.DataFrame([
    {'tenant': key.tenant_id, 'agent': key.agent_id, 'demand': node.demand_tokens,
     'grant': node.grant_tokens, 'deficit_priority': node.deficit_priority}
    for key, node in plan.allocation.agent_nodes.items()
])
decisions = pd.DataFrame([
    {'session': record.demand.dedup_key, 'cost': record.demand.total_cost_tokens,
     'selected': record.selected, 'reason': record.reason, 'rank': record.rank_hash[:10]}
    for record in (*plan.selection.selected, *plan.selection.unselected)
])
schedule = pd.DataFrame([
    {'request_id': item.request_id, 'session': item.session_id,
     'reserved_tokens': item.reserved_tokens, 'scheduled_seconds': item.scheduled_offset_seconds}
    for item in plan.schedule
])
display(grants.sort_values(['tenant', 'agent']))
display(decisions.sort_values(['selected', 'session'], ascending=[False, True]))
display(schedule)

,tenant,agent,demand,grant,deficit_priority
0,contoso,sales,250,215,0
1,contoso,support,300,256,0
2,fabrikam,support,500,429,100


,session,cost,selected,reason,rank
2,contoso/sales/alpha-2/v1,250,True,selected_after_redistribution,e15b1b2778
0,fabrikam/support/beta-1/v1,350,True,selected_within_initial_grant,19d3fa9dad
1,fabrikam/support/late-1/v1,150,True,selected_after_redistribution,7846dd5600
3,contoso/support/alpha-1/v1,300,False,too_large_for_agent_grant,3b4d2e0145


,request_id,session,reserved_tokens,scheduled_seconds
0,d5210a2259c153631e6f5bd1,alpha-2,250,0.0
1,d1c2a52ba06371c7a36502a8,beta-1,350,0.0
2,496a91bfa56db3c769be6a54,late-1,150,0.0


## 6. Record telemetry and the next fairness state

The telemetry reports selection utilization, coverage, slack, grant fairness, and TPM compliance. The output fairness deficits carry unserved demand into a later allocation; capacity itself does not roll over.

In [7]:
telemetry = build_batch_telemetry(
    allocation=plan.allocation, selection=plan.selection,
    total_eligible_sessions=len(plan.frame.sessions), tpm_compliance=True,
)
print(telemetry)
print('Next tenant deficits:', dict(plan.fairness_state_out.tenant_deficit_tokens))
print('Next agent deficits:', dict(plan.fairness_state_out.agent_deficit_tokens))

BatchTelemetry(utilization=0.8333333333333334, coverage=0.75, slack_tokens=150, selected_count=3, fairness_jain=0.9127727331120141, zero_allocations=0, tpm_compliance=True)
Next tenant deficits: {'contoso': 300, 'fabrikam': 50}
Next agent deficits: {'contoso/sales': 0, 'contoso/support': 300, 'fabrikam/support': 50}


## 7. Persist and commit the prepared plan

This uses the repository's single-process JSON reference store in a temporary directory. In production, the same lifecycle needs a durable shared transactional backend. Only a successful commit advances the watermark.

In [8]:
checkpoint_dir = TemporaryDirectory()
store = JsonCheckpointStore(Path(checkpoint_dir.name))
checkpoint = BatchCheckpoint(
    pipeline_id=plan.pipeline_id, batch_id=plan.batch_id, status='PREPARED',
    previous_successful_watermark=window.previous_successful_watermark, cutoff=window.cutoff,
    elapsed_minutes=window.elapsed_minutes, nominal_budget_tokens=budget.nominal_tokens,
    effective_budget_tokens=budget.effective_tokens, seed=plan.seed, frame_hash=plan.frame_hash,
    config_hash=plan.config_hash, membership_hash=plan.membership_hash,
    selected_ids=plan.selection.selected_ids, planned_usage_tokens=plan.planned_usage_tokens,
    actual_usage_tokens=0, retry_count=0,
    fairness_state={'tenant': dict(plan.fairness_state_out.tenant_deficit_tokens),
                    'agent': dict(plan.fairness_state_out.agent_deficit_tokens)},
    created_at=cutoff,
)
prepared = store.prepare(checkpoint, frame_hash=plan.frame_hash, config_hash=plan.config_hash, seed=plan.seed)
store.mark_running(prepared.batch_id)
store.settle(prepared.batch_id, actual_usage_tokens=plan.planned_usage_tokens)
store.commit(prepared.batch_id, success=True, new_watermark=window.cutoff, expected_previous_watermark=None)
print({'status': store.get(prepared.batch_id).status, 'watermark': store.latest_successful_watermark()})
checkpoint_dir.cleanup()

{'status': 'COMMITTED', 'watermark': datetime.datetime(2026, 8, 6, 12, 3, tzinfo=datetime.timezone.utc)}


## 8. Verify deterministic replay

A retry uses the same frozen frame, seed, budget, and fairness input. The selected membership and hashes must therefore match exactly.

In [9]:
replayed_plan = build_batch_plan(
    pipeline_id='walkthrough', batch_id='example-batch-001-replay', seed='walkthrough-seed-2026-08-06',
    window=window, budget=budget, frame=frame, fairness_state=fairness_in,
    allocation_config=AllocationConfig(tenant_floor_tokens=100, agent_floor_tokens=50),
)
assert replayed_plan.selection.selected_ids == plan.selection.selected_ids
assert replayed_plan.frame_hash == plan.frame_hash
assert replayed_plan.config_hash == plan.config_hash
print('Replay verified:', replayed_plan.selection.selected_ids)

Replay verified: ('contoso/sales/alpha-2/v1', 'fabrikam/support/beta-1/v1', 'fabrikam/support/late-1/v1')


## Summary

The walkthrough shows the full planning chain: frozen time window, explicit budget deductions, canonical eligible membership, fair token grants, deterministic whole-session selection, paced reservations, audited telemetry, durable checkpoint transitions, and replay verification.